In [0]:
import re
from datetime import datetime

def archive_sftp_files(sftp_path: str, archive_path: str):
    """
    Moves OLD files from the SFTP landing zone to archive folder
    when NEW files are uploaded.
    Keeps only the LATEST dated files in SFTP for processing.
    """
    try:
        files = dbutils.fs.ls(sftp_path)
        
        if not files:
            print(f"ℹ️ No files found in {sftp_path}. SFTP zone is empty.")
            return
        
       
        files_by_date = {}
        unparseable_files = []
        
        for file in files:
            if file.isDir():
                continue
            
            date_match = re.search(r'_(\d{8})', file.name)
            
            if date_match:
                date_str = date_match.group(1)
                
                try:
                    file_date = datetime.strptime(date_str, '%d%m%Y').date()
                    
                    if file_date not in files_by_date:
                        files_by_date[file_date] = []
                    files_by_date[file_date].append(file)
                    
                except ValueError:
                    print(f"⚠️ Could not parse date from filename: {file.name}")
                    unparseable_files.append(file)
                    continue
            else:

                print(f"⚠️ No date pattern found in filename: {file.name}")
                unparseable_files.append(file)
                continue
        
        if not files_by_date:
            print(f"ℹ️ No files with valid date patterns found. Skipping archival.")
            return
        
        latest_date = max(files_by_date.keys())
        latest_files = files_by_date[latest_date]
        
        old_files = []
        for file_date, file_list in files_by_date.items():
            if file_date < latest_date:
                old_files.extend(file_list)
        
        total_files = sum(len(file_list) for file_list in files_by_date.values())
        
        print(f"📂 Total files in SFTP      : {total_files}")
        print(f"📅 Latest date found        : {latest_date.strftime('%d-%m-%Y')}")
        print(f"🆕 Files with latest date   : {len(latest_files)}")
        print(f"📦 Old files to archive     : {len(old_files)}")
        
        if unparseable_files:
            print(f"⚠️  Unparseable files       : {len(unparseable_files)}")
        
        if not old_files:
            print(f"\n✅ {len(latest_files)} file(s) with latest date ({latest_date.strftime('%d-%m-%Y')}). No old files to archive.")
            return
        
        timestamp_folder  = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
        target_archive_dir = f"{archive_path}{timestamp_folder}/"
        
        print(f"\n📦 Archiving to: archive/{timestamp_folder}/")
        print(f"{'='*50}")
        
        archived_count = 0
        failed_count   = 0
        
        for file in old_files:
            try:
                source      = file.path
                destination = f"{target_archive_dir}{file.name}"
                
                file_mod_time = datetime.fromtimestamp(file.modificationTime / 1000)
                
                date_match = re.search(r'_(\d{8})', file.name)
                file_date_str = datetime.strptime(date_match.group(1), '%d%m%Y').strftime('%d-%m-%Y') if date_match else 'unknown'
                
                dbutils.fs.mv(source, destination)
                
                print(f"  ✅ Archived : {file.name}")
                print(f"     File Date: {file_date_str}")
                print(f"     Modified : {file_mod_time.strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"     Moved to : archive/{timestamp_folder}/")
                archived_count += 1
            
            except Exception as e:
                failed_count += 1
                print(f"  ❌ FAILED   : {file.name}")
                print(f"     Error    : {str(e)}")
                print(f"\n🚨 ALERT: Archival failed for {file.name} — manual check needed!")
        
        print(f"\n{'='*50}")
        print(f"📊 ARCHIVAL SUMMARY")
        print(f"{'='*50}")
        print(f"  ✅ Successfully archived        : {archived_count} file(s)")
        print(f"  ❌ Failed                       : {failed_count} file(s)")
        print(f"  🆕 Latest files kept in SFTP    : {len(latest_files)} file(s) ({latest_date.strftime('%d-%m-%Y')})")
        
        if failed_count > 0:
            print(f"\n🚨 ALERT: {failed_count} file(s) failed to archive!")
            print(f"🚨 Please check archive folder manually!")
    
    except Exception as e:
        if "java.io.FileNotFoundException" in str(e):
            print(f"ℹ️ Path {sftp_path} does not exist. Skipping.")
        else:
            print(f"🚨 CRITICAL ALERT: Archival process failed!")
            print(f"   Error: {str(e)}")

In [0]:
# S3 paths
S3_BUCKET         = "s3a://retail-sales-dwh-sadhvika"
sftp_zone         = f"{S3_BUCKET}/sftp/"
archive_destination = f"{S3_BUCKET}/archive/"

print("="*50)
print("   SFTP ARCHIVAL PROCESS STARTED")
print(f"   Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*50)

archive_sftp_files(sftp_zone, archive_destination)

print("\n✅ SFTP Archival Process Complete!")

In [0]:
print("=== Files remaining in SFTP (latest only) ===")
try:
    sftp_files = dbutils.fs.ls(f"{S3_BUCKET}/sftp/")
    for f in sftp_files:
        mod_time = datetime.fromtimestamp(f.modificationTime / 1000)
        print(f"  📄 {f.name} | Modified: {mod_time.strftime('%Y-%m-%d %H:%M:%S')}")
except Exception as e:
    print(f"❌ Error: {str(e)}")

print("\n=== Files in Archive folder ===")
try:
    archive_folders = dbutils.fs.ls(f"{S3_BUCKET}/archive/")
    for folder in archive_folders:
        print(f"  📁 {folder.name}")
        archive_files = dbutils.fs.ls(folder.path)
        for af in archive_files:
            print(f"      📦 {af.name}")
except Exception as e:
    print(f"❌ Error: {str(e)}")

In [0]:
print("\n=== ARCHIVAL VALIDATION REPORT ===\n")

TABLE_PREFIXES = [
    "customers_src",
    "products_src",
    "stores_src",
    "sales_transactions_src"
]

all_passed = True

try:
    sftp_files = dbutils.fs.ls(f"{S3_BUCKET}/sftp/")
    
    for prefix in TABLE_PREFIXES:
        matched = [f for f in sftp_files 
                   if f.name.startswith(prefix) 
                   and f.name.endswith('.csv')]
        
        print(f"Table Prefix : {prefix}")
        print(f"Files in SFTP: {len(matched)}")
        
        if len(matched) == 1:
            print(f"✅ PASS — Only 1 (latest) file present in SFTP")
        elif len(matched) == 0:
            print(f"⚠️  WARNING — No files found in SFTP")
            all_passed = False
        else:
            print(f"❌ FAIL — {len(matched)} files still in SFTP!")
            all_passed = False
        print()

except Exception as e:
    print(f"❌ Validation error: {str(e)}")
    all_passed = False

print("="*40)
if all_passed:
    print("✅ ALL VALIDATIONS PASSED")
else:
    print("❌ SOME VALIDATIONS FAILED — Check archive!")
print("="*40)